# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset on second primary colorectal cancer survivors using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their names and IDs.

In [ ]:
# List all record sets available in the dataset and show their @id
recordset_ids = []
print("Record sets in the dataset:")
for recordset in dataset.recordsets():
    print(f"- Name: {recordset.name} | @id: {recordset.id}")
    recordset_ids.append(recordset.id)
    # Show their fields/columns by @id
    print("  Fields:")
    for field in recordset.fields:
        # Use 'id' attribute for @id
        print(f"    - Name: {field.name} | @id: {field.id}")
    print("")
if not recordset_ids:
    print('  (No record sets found at the top-level, please check for nested content or Croissant schema changes.)')

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Reference each item by its `@id` as shown in the overview above.

In [ ]:
# Extract all available record sets (by @id) to DataFrames
if not recordset_ids:
    print('No record sets available to extract.')
else:
    dataframes = {}
    for recordset_id in recordset_ids:
        print(f"Loading records for record set: {recordset_id}")
        records = list(dataset.records(record_set=recordset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[recordset_id] = df
            print(f"Loaded {len(df)} rows. Columns:", df.columns.tolist())
        else:
            print('  No records found.')
    if dataframes:
        # Show one DataFrame as example (choose the first loaded)
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nSample records from record set {first_rs_id}:")
        display(dataframes[first_rs_id].head())
    else:
        print('No tabular data found in any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for summary statistics. Entities (fields/columns) must be referenced by their `@id`.

In [ ]:
# Example: Identify a numeric field for EDA
# You'll typically want to select a field like 'age' or a continuous variable by @id, such as 'age_at_second_primary' if available
# Let's attempt to discover numeric fields in the first DataFrame:

if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Columns for EDA in record set {first_rs_id}:\n", df.columns.tolist())

    # Try to auto-detect a numeric column
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Try to force-cast columns to numeric if possible
        for col in df.columns:
            try:
                df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
                if df[col+'_num'].notnull().sum() > 0:
                    numeric_field_id = col+'_num'
                    break
            except Exception:
                continue
    
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field, such as 'Sex' or similar, by `@id`
        group_field = None
        for col in df.columns:
            if (df[col].nunique() > 1 and df[col].nunique() < len(df)//2 and
                not pd.api.types.is_numeric_dtype(df[col])):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field detected for EDA in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Refer to fields by their `@id`.

In [ ]:
# Example: Visualize the distribution of a (numeric) field and compare across a group, referencing by @id
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if dataframes and numeric_field_id:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {first_rs_id}")
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, overview, and explore the FAIR² colorectal cancer survivors dataset using the Croissant schema and the `mlcroissant` library. All fields and record sets were referenced via their `@id` values as per FAIR best practices.

- We listed record sets and their field `@id`s for transparent referencing.
- We loaded data into DataFrames and performed rudimentary EDA operations such as filtering, normalization, and grouping, referencing columns by `@id`.
- We visualized distributions and relationships to support initial insights.

You can now further explore this dataset or adapt the code for more in-depth analysis, always referencing entities by their `@id` as established.

**For reproducibility and maintainability, keep referencing all content by `@id` in your analysis and documentation.**